In [1]:
from pathlib import Path
from collections import Counter

dataset_path = Path("../data/raw/GVLiD/DATASET")

image_extensions = {".jpg", ".jpeg", ".png"}

class_names = [
    "healthy",
    "esca",
    "Black rot",
    "leaf blight"
]

for class_name in class_names:

    folder = dataset_path / class_name

    images = [
        f for f in folder.iterdir()
        if f.is_file()
        and f.suffix.lower() in image_extensions
    ]

    print(f"{class_name}: {len(images)} images")

healthy: 1109 images
esca: 888 images
Black rot: 808 images
leaf blight: 672 images


In [2]:
total_images = 0

for class_name in class_names:
    folder = dataset_path / class_name

    count = sum(
        1 for f in folder.iterdir()
        if f.is_file()
        and f.suffix.lower() in image_extensions
    )

    total_images += count

print("Total images:", total_images)

Total images: 3477


In [3]:
import json
from pathlib import Path
from collections import Counter

dataset_path = Path("../data/raw/GVLiD/DATASET")

with open(dataset_path / "metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("Total metadata records:", len(metadata))

# Count disease labels from metadata
metadata_counts = Counter(
    record["Disease_Type"] for record in metadata
)

print("\nMetadata class distribution:")
for label, count in metadata_counts.items():
    print(label, ":", count)

Total metadata records: 3477

Metadata class distribution:
Black Rot : 809
Esca : 888
Healthy : 1109
Leaf Blight : 671


In [5]:
from pathlib import Path
from collections import defaultdict

dataset_path = Path("../data/raw/GVLiD/DATASET")

folder_mapping = {
    "healthy": "Healthy",
    "esca": "Esca",
    "Black rot": "Black Rot",
    "leaf blight": "Leaf Blight"
}

# Create filename-to-folder mapping
image_locations = defaultdict(list)

for folder_name, class_label in folder_mapping.items():

    folder_path = dataset_path / folder_name

    for image_path in folder_path.iterdir():

        if image_path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
            image_locations[image_path.name].append(class_label)

missing_images = []
label_mismatches = []
duplicate_images = []

for record in metadata:

    filename = record["image_filename"]
    metadata_label = record["Disease_Type"]

    locations = image_locations.get(filename, [])

    if not locations:
        missing_images.append(filename)

    elif len(locations) > 1:
        duplicate_images.append(filename)

    elif locations[0] != metadata_label:
        label_mismatches.append({
            "filename": filename,
            "folder_label": locations[0],
            "metadata_label": metadata_label
        })

print("Total metadata records:", len(metadata))
print("Missing images:", len(missing_images))
print("Label mismatches:", len(label_mismatches))
print("Duplicate filenames:", len(duplicate_images))

print("\nFirst 20 label mismatches:")
for item in label_mismatches[:20]:
    print(item)

Total metadata records: 3477
Missing images: 0
Label mismatches: 1344
Duplicate filenames: 0

First 20 label mismatches:
{'filename': 'esca001.jpg', 'folder_label': 'Esca', 'metadata_label': 'Black Rot'}
{'filename': 'leaf blight001.jpg', 'folder_label': 'Leaf Blight', 'metadata_label': 'Esca'}
{'filename': 'leaf blight002.jpg', 'folder_label': 'Leaf Blight', 'metadata_label': 'Healthy'}
{'filename': 'leaf blight003.jpg', 'folder_label': 'Leaf Blight', 'metadata_label': 'Healthy'}
{'filename': 'leaf blight004.jpg', 'folder_label': 'Leaf Blight', 'metadata_label': 'Healthy'}
{'filename': 'leaf blight005.jpg', 'folder_label': 'Leaf Blight', 'metadata_label': 'Healthy'}
{'filename': 'leaf blight006.jpg', 'folder_label': 'Leaf Blight', 'metadata_label': 'Healthy'}
{'filename': 'leaf blight007.jpg', 'folder_label': 'Leaf Blight', 'metadata_label': 'Healthy'}
{'filename': 'leaf blight008.jpg', 'folder_label': 'Leaf Blight', 'metadata_label': 'Healthy'}
{'filename': 'leaf blight009.jpg', 'fol

In [6]:
import pandas as pd

df = pd.DataFrame(metadata)

df["filename_class"] = (
    df["image_filename"]
    .str.lower()
    .str.extract(r"^(black rot|esca|healthy|leaf blight)", expand=False)
)

df["filename_class"] = df["filename_class"].replace({
    "black rot": "Black Rot",
    "leaf blight": "Leaf Blight",
    "esca": "Esca",
    "healthy": "Healthy"
})

comparison = pd.crosstab(
    df["filename_class"],
    df["Disease_Type"],
    margins=True
)

display(comparison)

Disease_Type,Black Rot,Esca,Healthy,Leaf Blight,All
filename_class,,,,,
Black Rot,807,0,0,0,807
Esca,1,887,0,0,888
Healthy,0,0,438,671,1109
Leaf Blight,0,1,671,0,672
All,808,888,1109,671,3476


In [7]:
# Find records where filename prefix could not be extracted

unmatched_prefix = df[df["filename_class"].isna()]

print("Unmatched prefix records:")
display(unmatched_prefix[
    ["Image_ID", "image_filename", "Disease_Type"]
])

Unmatched prefix records:


,Image_ID,image_filename,Disease_Type
807,IMG_0808,mega dataset (31) - Copy - Copy - Copy.jpg,Black Rot


In [8]:
# Build a dataframe using actual folder locations

folder_records = []

for folder_name, folder_label in folder_mapping.items():

    folder_path = dataset_path / folder_name

    for image_path in folder_path.iterdir():

        if image_path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
            folder_records.append({
                "image_filename": image_path.name,
                "folder_label": folder_label
            })

folder_df = pd.DataFrame(folder_records)

validation_df = df.merge(
    folder_df,
    on="image_filename",
    how="left",
    validate="one_to_one"
)

print("Total joined records:", len(validation_df))

display(pd.crosstab(
    validation_df["folder_label"],
    validation_df["Disease_Type"],
    margins=True
))

Total joined records: 3477


Disease_Type,Black Rot,Esca,Healthy,Leaf Blight,All
folder_label,,,,,
Black Rot,808,0,0,0,808
Esca,1,887,0,0,888
Healthy,0,0,438,671,1109
Leaf Blight,0,1,671,0,672
All,809,888,1109,671,3477
